# Stanowisko 4 - napęd omni (swerve drive) - kinematyka prosta

### Do zweryfikowania poprawności połączenia notatnika z robotem można wysłać na jedno z kół prędkości sterujące. 

Poniższe polecenie na temacie **/cmd_joint_states** opublikuje wiadomość typu **JointState** z pakietu **sensor_msgs**

W celu zatrzymania działającego programu w komórce **(oznaczenie * )** należy wcisnąć przycisk **interrupt the kernel (czarny kwadraty na górnym pasku)**


Po zweryfikowaniu prawidłowej komunikacji notatnika z robotem można na początku lini wstawić komentarz **#** , aby zablokować jej uruchamianie. Po wysłaniu polecenia jedno z kół powinno się kręcić.


In [ ]:
# ! lub %%bash umożliwia wywołanie w notatniku polecenia, które uruchamia się w konsoli (terminalu).

# Wysłanie prędkości na jedno z kół - jeśli dalsze uruchomienie nie jest potrzebne to wstawić znak komentarza #
# Ponisze polecenie powoduje uruchomienie nieskończonej pętli i uniemożliwia wywołanie kolejnych poleceń.
! ros2 topic pub -r 1 /cmd_joint_states sensor_msgs/msg/JointState "{velocity: [0.1, 0.0, 0.0, 0.0]}"
# ! ros2 topic pub -r 1 /cmd_joint_states sensor_msgs/msg/JointState "{velocity: [0.1, 0.0, 0.0, 0.0]}"

In [ ]:
# Zatrzymanie kół robota - wysłanie prędkości 0.0
# Jeśli dalsze uruchomienie nie jest potrzebne to wstawić znak komentarza #
# Ponisze polecenie powoduje uruchomienie nieskończonej pętli i uniemożliwia wywołanie kolejnych poleceń.
! ros2 topic pub -r 1 /cmd_joint_states sensor_msgs/msg/JointState "{velocity: [0.0, 0.0, 0.0, 0.0]}"

---

## Import podstawowych bibliotek ROS2

In [ ]:
# Główna biblioteka ROS2 dla Pythona – pozwala na uruchamianie i obsługę nodów (programów w ROS2)
import rclpy 

# Klasa bazowa do tworzenia własnych nodów – każdy node w ROS2 dziedziczy po tej klasie
from rclpy.node import Node

# Import standardowej wiadomości JointState z pakietu sensor_msgs dla ROS2. Wiadomość służy do publikowania 
# i subskrybowania stanu przegubów (złącz) (np. pozycja, prędkość, moment obrotowy)
from sensor_msgs.msg import JointState

# Wiadomość ROS2 z pakietu nav_msgs opisująca odometrię robota.
# Zawiera informację o pozycji i orientacji (Pose) oraz prędkościach (Twist).
# Jest podstawowym sposobem przekazywania danych o ruchu i położeniu robota
# w systemach lokalizacji i nawigacji.
from nav_msgs.msg import Odometry

# Wiadomość ROS2 z pakietu geometry_msgs opisująca orientację w postaci kwaternionu (x, y, z, w).
# Kwaterniony to matematyczna reprezentacja obrotów w 3D, używana w odometrii i transformacjach.
# Stosowane są w celu uniknięcia problemów z kątem Eulera (np. gimbal lock).
from geometry_msgs.msg import Quaternion

## Import bibliotek do obsługi matematyki dla Python

In [ ]:
# Popularna biblioteka numeryczna w Pythonie – używana do obliczeń macierzowych i wektorowych
import numpy as np

# Standardowa biblioteka Pythona z funkcjami matematycznymi (np. sin, cos, sqrt)
import math

## <span style="color:red">Uzupełnij poniższą komórkę. (1/2)</span>

Wstaw parametry kinematyki wyznaczone w ćwiczeniu 1.

Parametry należy wykorzystać podczas dalszej implementacji wzorów.

---

In [ ]:
class Robot(Node):
    """
    Główna klasa programu, która obsługuje kinematykę prostą robota w ROS2.
    Kinematyka prosta polega na tym, że na podstawie prędkości obrotowych kół
    obliczane są prędkości liniowe i kątowe całego robota. Na podstawie tych prędkości wyznaczana jest
    aktualna pozycja i orientacja robota (odometria).

    Działanie:
    - Inicjalizuje węzeł ROS2 o nazwie 'robot_forward_kinematics'.
    - Subskrybuje temat /joint_states, aby odbierać aktualne prędkości kół.
    - Na podstawie odebranych danych oblicza przesunięcie i rotację robota (x, y, θ).
    - Publikuje wyliczoną odometrię w temacie /odometry jako wiadomość typu Odometry.
    """
    def __init__(self):
        # Inicjalizacja węzła ROS2 z unikalną nazwą 'robot_forward_kinematics'.
        # Węzeł to podstawowy element w ROS2, który może subskrybować i publikować dane.
        super().__init__('robot_forward_kinematics')
        
        # --- ZMIENNE ROBOTA ---
        # Współrzędne (x, y) i kąt theta służą do zapisywania bieżącej pozycji
        # i orientacji robota w przestrzeni. Są aktualizowane na podstawie prędkości kół.
        self.x = 0      # Pozycja robota w osi X (metry)
        self.y = 0      # Pozycja robota w osi Y (metry)
        self.theta = 0  # Orientacja robota (kąt wokół osi Z, radiany)

        # --- SUBSKRYPCJA prędkości kół ---
        # Subskrybujemy temat /joint_states, w którym publikowane są wiadomości typu JointState.
        # Zawierają one m.in. prędkości obrotowe kół robota, kąty skrętu tych kół.
        # Po odebraniu nowej wiadomości uruchamiana jest metoda joint_states_callback().
        self.subscription_joint_states = self.create_subscription(
            JointState,                 # Typ wiadomości (stan kół: pozycja, prędkość)
            '/joint_states',            # Nazwa tematu z danymi o prędkościach kół
            self.joint_states_callback, # Funkcja wywoływana po odebraniu wiadomości
            10                          # QoS profile depth – ile wiadomości może buforować
        )
        
        # --- PUBLIKACJA odometrii ---
        # Tworzymy publisher na temat /odometry. 
        # W tym temacie publikowana jest wiadomość Odometry, zawierająca:
        #  - bieżącą pozycję (x, y),
        #  - orientację (theta, w postaci kwaternionu),
        #  - prędkości liniowe i kątowe robota.
        # Dzięki temu inne węzły (np. system lokalizacji, wizualizacja w RViz)
        # mogą korzystać z wyliczonej odometrii.
        self.publisher_odom = self.create_publisher(
            Odometry,        # Typ wiadomości opisującej odometrię robota
            '/odometry',     # Nazwa tematu publikacji
            10               # QoS profile depth
        )

In [ ]:
# rclpy.ok() sprawdza, czy biblioteka rclpy jest już uruchomiona (czy działa "event loop").
# Jeżeli nie, wywołujemy rclpy.init(args=None), aby zainicjalizować środowisko ROS2 dla języka Python.
# Dzięki temu węzeł będzie mógł tworzyć subskrypcje, publishery i wykonywać callbacki.
if not rclpy.ok():
    rclpy.init(args=None)

---

<div style="text-align:center; font-size:24px; font-weight:bold;">Kinematyka prosta</div>

<img src="../images/prosta.png" alt="System architecture" width="800"/>

## Macierze dla robota z napędem omni (swerve drive)

### **Macierz $(\mathbf{A})$ – Ograniczenia kinematyczne**
$$
\mathbf{A} =
\begin{bmatrix}
\sin(\alpha_1 + \beta_1 + \theta_1)  & -\cos(\alpha_1 + \beta_1 + \theta_1) & -l_1 \cos(\beta_1 + \theta_1) \\
\sin(\alpha_2 + \beta_2 + \theta_2)  & -\cos(\alpha_2 + \beta_2 + \theta_2) & -l_2 \cos(\beta_2 + \theta_2) \\
\sin(\alpha_3 + \beta_3 + \theta_3)  & -\cos(\alpha_3 + \beta_3 + \theta_3) & -l_3 \cos(\beta_3 + \theta_3) \\
\sin(\alpha_4 + \beta_4 + \theta_4)  & -\cos(\alpha_4 + \beta_4 + \theta_4) & -l_4 \cos(\beta_4 + \theta_4) \\
\cos(\alpha_1 + \beta_1 + \theta_1)  & \sin(\alpha_1 + \beta_1 + \theta_1)  & l_1 \sin(\beta_1 + \theta_1) \\
\cos(\alpha_2 + \beta_2 + \theta_2)  & \sin(\alpha_2 + \beta_2 + \theta_2)  & l_2 \sin(\beta_2 + \theta_2) \\
\cos(\alpha_3 + \beta_3 + \theta_3)  & \sin(\alpha_3 + \beta_3 + \theta_3)  & l_3 \sin(\beta_3 + \theta_3) \\
\cos(\alpha_4 + \beta_4 + \theta_4)  & \sin(\alpha_4 + \beta_4 + \theta_4)  & l_4 \sin(\beta_4 + \theta_4)
\end{bmatrix}
$$

Gdzie:
- \( $\alpha_n$ \) – orientacja koła \( n \) w układzie robota (kąt położenia koła względem środka robota, liczony od osi X robota),
- \( $\beta_n$ \) – kąt skrętu koła,
- \( $\theta_n$ \) – obliczony kąt skrętu koła \( n \),
- \( $l_n$ \) – odległość koła od środka układu robota.

---

### **Macierz $(\mathbf{B})$ – Transformacja prędkości kół**
$$
\mathbf{B} =
\begin{bmatrix}
r & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & r & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & r & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & r & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0
\end{bmatrix}
$$

Gdzie:
- \( r \) – promień koła.

---

## Obliczanie prędkości robota $\mathbf{\dot{\xi}_{R}}$

Korzystając z równań kinematyki prostej:

$$
	\mathbf{\dot{\xi_{R}}} = (\mathbf{A}^T \mathbf{A})^{-1}\mathbf{A}^T \mathbf{B} \mathbf{\dot{\varphi}}
$$


Ostatecznie otrzymujemy wektor prędkości robota.

Ten wektor należy następnie (znając różnicę czasu między wiadomościami $\delta t$ wykorzystać do obliczenia pozycji robota na płaszczyźnie. 

---

##  Funkcja pomocniczna

In [ ]:
def theta_to_quaternion(theta):
    """
    Funkcja pomocnicza, która zamienia kąt obrotu robota w 2D (tzw. yaw = theta)
    na reprezentację kwaternionową. 
    W ROS2 orientacja robota w wiadomościach (np. Odometry, Pose) jest zawsze
    zapisywana jako kwaternion, nawet jeśli poruszamy się tylko w 2D.

    Dlaczego to potrzebne?
    - W 2D wystarczy jeden kąt obrotu (theta, yaw), aby opisać orientację robota.
    - Jednak wiadomości ROS2 (geometry_msgs/Quaternion) wymagają czterech współrzędnych
      (x, y, z, w), które opisują orientację w przestrzeni 3D.
    - Dlatego trzeba przeliczyć kąt theta na równoważny kwaternion.

    :param theta (float): Kąt obrotu robota wokół osi Z w radianach.

    :return Quaternion: Kwaternion odpowiadający podanemu kątowi yaw.

    Zastosowanie praktyczne:
    - Funkcja przydaje się podczas obliczania odometrii robota.
    - Pozwala wypełnić pole orientation w wiadomości Odometry lub Pose,
      które później można zwizualizować np. w RViz.
    """
    q = Quaternion()
    q.z = np.sin(theta / 2.0)  # Składowa odpowiadająca obrotowi wokół osi Z
    q.w = np.cos(theta / 2.0)  # Składowa normalizująca kwaternion
    return q

## <span style="color:red">Uzupełnij poniższą komórkę. (2/2)</span>

Należy uzupełnić funkcję `joint_states_callback`, tak aby realizowała funkcjonalność **kinematyki prostej**.

1. **Wyznaczyć różnicę czasu** $\delta t$ między kolejnymi wiadomościami przekazywanymi do `joint_states_callback`.

   Wiadomość ta nie zawiera uzupełnionych znaczników czasu, na podstawie których najlepiej byłoby obliczać różnicę czasu $\delta t$ pomiędzy kolejnymi wiadomościami. Z tego powodu różnicę czasu można obliczyć na dwa sposoby:

   - **Opcja I** – wykorzystać bibliotekę `time` w Pythonie i obliczyć różnicę czasu pomiędzy kolejnymi odebranymi wiadomościami:

     ```python
     import time		
     t1 = time.time()
     # ... coś się dzieje ...
     t2 = time.time()
     dt = t2 - t1   # różnica w sekundach (float)
     print(f"Różnica czasu: {dt:.3f} s")
     ```

   - **Opcja II** – wersja uproszczona: sprawdzić, z jaką częstotliwością publikowana jest wiadomość na temacie `/joint_states`.  
     Jest to polecenie używane w terminalu ROS, dlatego należy uruchomić je w nowym terminalu lub w **Jupyter Notebook**, poprzedzając komendę znakiem `!`. Zalecane jest jednak użycie poniższego polecenia bezpośrednio w terminalu:

     ```bash
     ros2 topic hz /joint_states
     ```

---

2. **Obliczyć wektor prędkości** $\dot{\xi}_R$ robota na podstawie prędkości kół $\mathbf{\dot{\phi}}$ oraz kątów skrętu $\mathbf{\theta_{kolo}}$ (jeśli występują):

   $$
	\mathbf{\dot{\xi_{R}}} = (\mathbf{A}^T \mathbf{A})^{-1}\mathbf{A}^T \mathbf{B} \mathbf{\dot{\varphi}}
   $$

---

3. **Obliczyć pozycję $(x, y)$ oraz orientację $\theta_{robot}$ robota** na płaszczyźnie na podstawie wyznaczonych prędkości z poprzedniego punktu.  

   Należy pamiętać, że występują tutaj dwa układy współrzędnych:
   - globalny (związany z płaszczyzną),
   - lokalny (związany z robotem).

   Następnie należy zaktualizować pola klasy:

   ```python
   self.x = 0      # Pozycja robota w osi X (metry)
   self.y = 0      # Pozycja robota w osi Y (metry)
   self.theta = 0  # Orientacja robota (kąt obrotu wokół osi Z, radiany)
   ```
---
4. **Uzupełnić wiadomość `odom_msg`**, wykorzystując wyznaczone w poprzednim punkcie wartości pozycji robota w osiach $X$, $Y$ oraz jego orientację $\theta_{robot}$.
---
5. **Zweryfikować poprawność wyników na rzeczywistym robocie.**  
   Zaproponować testy umożliwiające ocenę działania i dokładności odometrii.  
   Sprawdzenie można przeprowadzić np. w **RViz** — po przejechaniu przez robota kwadratu w rzeczywistości trajektoria odometrii powinna przypominać kwadrat, a nie linię.
> ⚠️ **Zebrać dane do sprawozdania.**

In [ ]:
def joint_states_callback(self, msg):
    """
    Callback funkcji odbierającej dane o prędkościach kół i obliczającej odometrię robota.

    :param msg: Wiadomość typu sensor_msgs.msg.JointState zawierająca:
        - msg.velocity: lista prędkości kątowych kół w rad/s w ustalonej kolejności (wheel0, wheel1, wheel2...)
        - msg.position (opcjonalnie): lista kątów skrętu kół w radianach, jeśli robot ma skrętne koła

    Pożądane wyjście:
    - Wiadomość typu nav_msgs.msg.Odometry, zawierająca:
        - odometry.pose.pose: pozycję (x, y) i orientację (quaternion) robota
        - odometry.twist.twist: prędkość liniową (linear) i kątową robota w układzie bazowym
    """
    ## 1. UZUPEŁNIJ - wyznacz delta t
    
    ## 2. UZUPEŁNIJ - wyznacz prędkości robota z równań kinematyki prostej  
    
    ## 3. UZUPEŁNIJ - wyznacz i uzupełnij pozycję i orientację robota

    # Pozycja robota w osi X (metry)
    self.x
    
    # Pozycja robota w osi Y (metry)
    self.y
    
    # Orientacja robota (kąt obrotu wokół osi Z, radiany)
    self.theta
    
    odom_msg = Odometry()
    odom_msg.header.frame_id = "odom"
    ## 4 UZUPEŁNIJ - poprawnymi danymi pola wiadomości:
    odom_msg.pose.pose.position.x =
    odom_msg.pose.pose.position.y =
    odom_msg.pose.pose.orientation =
    odom_msg.twist.twist.linear.x =
    odom_msg.twist.twist.linear.y =
    odom_msg.twist.twist.angular.z =

    # Publikacja wiadomości typu Odometry na temacie /odometry
    # self.publisher_odom – publisher zdefiniowany w konstruktorze klasy Robot,
    # który posiada metodę publish. Metoda ta jako argument przyjmuje obiekt (wiadomość) ROS2.
    # Typ wiadomości zależy od argumentów przekazanych w konstruktorze.
    # W tym przypadku przekazany obiekt odom_msg jest typu Odometry i zawiera
    # obliczoną odometrię robota (pozycję x, y, orientację theta w postaci kwaternionu,
    # oraz prędkości liniowe i kątowe).
    self.publisher_odom.publish(odom_msg)

In [ ]:
# Robot.joint_states_callback = joint_states_callback
#
# Ta konstrukcja przypisuje wcześniej zdefiniowaną funkcję `joint_states_callback`
# do klasy `Robot` jako jej metodę.
#
# Dzięki temu obiekty klasy Robot mogą korzystać z tej funkcji tak,
# jakby była napisana bezpośrednio wewnątrz klasy:
#     robot = Robot()
#     robot.joint_states_callback(msg)
#
# Jest to sposób „doklejenia” funkcji do klasy po jej zdefiniowaniu.
# W tym przypadku metoda obsługuje wiadomości z tematu /joint_states,
# czyli dane o aktualnych prędkościach i pozycji kół robota.
Robot.joint_states_callback = joint_states_callback

In [ ]:
def main(args=None):
    # Tworzy instancję noda = uruchamia węzeł ROS2 (zdefiniowany w klasie Robot)
    # W tym momencie dostępne są już subskrypcje i publishery.
    node = Robot()

    # Główna pętla programu: dopóki system ROS2 działa,
    # obsługiwane są wszystkie przychodzące wiadomości i callbacki.
    # spin_once() wykonuje jedną iterację pętli zdarzeń, w odróżnieniu od spin(),
    # który działa w trybie ciągłym i blokującym.
    while rclpy.ok():
        rclpy.spin_once(node)

    # Można opcjonalnie usunąć noda ręcznie:
    # node.destroy_node()

    try:
        # Alternatywny sposób: rclpy.spin(node) – zwykle prostszy, ale mniej elastyczny.
        pass
    finally:
        # Bezpieczne zamknięcie ROS2 nawet przy błędach lub przerwaniu programu.
        if rclpy.ok():
            rclpy.shutdown()

# Standardowy idiom w Pythonie: uruchomienie main() tylko wtedy,
# gdy ten plik jest wykonywany bezpośrednio (a nie importowany).
# Uruchomienie funkcji głównej po uruchomieniu skryptu
if __name__ == '__main__':
    main()